In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

# ==========================================
# Task 1: Download & Preprocess Data
# ==========================================
# Change this variable to 'cifar100' to train on the other dataset
DATASET_NAME = 'fashion_mnist'

print(f"Loading and preprocessing {DATASET_NAME} dataset...")

if DATASET_NAME == 'fashion_mnist':
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
    # Normalize pixel values to be between 0 and 1
    x_train, x_test = x_train / 255.0, x_test / 255.0
    # Add channel dimension because Fashion-MNIST is grayscale (28x28 -> 28x28x1)
    x_train = np.expand_dims(x_train, -1)
    x_test = np.expand_dims(x_test, -1)
    num_classes = 10
    input_shape = (28, 28, 1)
elif DATASET_NAME == 'cifar100':
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()
    # Normalize pixel values
    x_train, x_test = x_train / 255.0, x_test / 255.0
    num_classes = 100
    input_shape = (32, 32, 3) # CIFAR is RGB
    # Flatten target labels for sklearn metrics later
    y_train = y_train.flatten()
    y_test = y_test.flatten()

print(f"Training data shape: {x_train.shape}")
print(f"Testing data shape: {x_test.shape}")

# ==========================================
# Task 2: Implement CNN Architecture
# ==========================================
def build_model(input_shape, num_classes):
    model = models.Sequential([
        # Convolutional Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),

        # Convolutional Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Convolutional Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Fully Connected Layers
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4), # Prevents overfitting
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_model(input_shape, num_classes)
print("\nModel Architecture Summary:")
model.summary()

# ==========================================
# Task 3: Train the Model (Cross-entropy & Adam)
# ==========================================
# Task 5: Hyperparameters (Modify these to experiment)
LEARNING_RATE = 0.001
BATCH_SIZE = 64
EPOCHS = 10

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("\nStarting Training Process...")
history = model.fit(x_train, y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(x_test, y_test))

# ==========================================
# Task 4 & 6: Evaluation & Visualization
# ==========================================
# Plotting Training and Validation Loss & Accuracy
plt.figure(figsize=(14, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
plt.title(f'Accuracy over Epochs ({DATASET_NAME})')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title(f'Loss over Epochs ({DATASET_NAME})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()

print("\nEvaluating Model on Test Data...")
test_loss, test_acc = model.evaluate(x_test,  y_test, verbose=0)
print(f"Final Test Accuracy: {test_acc*100:.2f}%\n")

# Generate predictions for Precision, Recall, and F1-score
print("Generating Detailed Classification Report...")
y_pred_probs = model.predict(x_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Sklearn's classification_report calculates precision, recall, and f1-score automatically
report = classification_report(y_test, y_pred_classes)
print(report)

Loading and preprocessing fashion_mnist dataset...
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training data shape: (60000, 28, 28, 1)
Testing data shape: (10000, 28, 28, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Model Architecture Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,410 (1.49 MB)

 Trainable params: 390,410 (1.49 MB)

 Non-trainable params: 0 (0.00 B)


Starting Training Process...
Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 98s 102ms/step - accuracy: 0.8206 - loss: 0.4936 - val_accuracy: 0.8766 - val_loss: 0.3440
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 95s 101ms/step - accuracy: 0.8881 - loss: 0.3062 - val_accuracy: 0.9001 - val_loss: 0.2722
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 142s 102ms/step - accuracy: 0.9068 - loss: 0.2575 - val_accuracy: 0.9042 - val_loss: 0.2603
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 94s 100ms/step - accuracy: 0.9176 - loss: 0.2225 - val_accuracy: 0.9104 - val_loss: 0.2439
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 143s 102ms/step - accuracy: 0.9230 - loss: 0.2049 - val_accuracy: 0.9163 - val_loss: 0.2283
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 94s 100ms/step - accuracy: 0.9338 - loss: 0.1803 - val_accuracy: 0.9164 - val_loss: 0.2434
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 95s 102ms/step - accuracy: 0.9383 - loss: 0.1649 - val_accuracy: 0.9190 - val_loss: 0.2307
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 94s 100ms/s